# ECON 3385 - Problem Set 7

Anton Melnychuk

### Question 1

The market share of airline $j$ in route-quarter $(c,t)$ can be written as:

$$s_{jct} = s_{j|G,ct} \cdot s_{G,ct}$$

where:
- $s_{j|G,ct}$ is the conditional share of airline $j$ within nest $G$ (given that nest $G$ is chosen)
- $s_{G,ct}$ is the share of the inside nest $G$ (probability of choosing any airline)

The conditional share of airline $j$ within nest $G$ is

$$s_{j|G,ct} = \frac{\exp(\delta_{jct} / \lambda)}{\sum_{k \in G} \exp(\delta_{kct} / \lambda)}$$

where $\delta_{jct} = \beta_j + \alpha p_{jct} + \xi_{jct}$ is the mean utility.

The share of the inside nest $G$ is

$$s_{G,ct} = \frac{\left(\sum_{k \in G} \exp(\delta_{kct} / \lambda)\right)^\lambda}{1 + \left(\sum_{k \in G} \exp(\delta_{kct} / \lambda)\right)^\lambda}$$

Combining these, the market share of airline $j$ is

$$s_{jct} = \frac{\exp(\delta_{jct} / \lambda)}{\sum_{k \in G} \exp(\delta_{kct} / \lambda)} \cdot \frac{\left(\sum_{k \in G} \exp(\delta_{kct} / \lambda)\right)^\lambda}{1 + \left(\sum_{k \in G} \exp(\delta_{kct} / \lambda)\right)^\lambda}$$

$$s_{jct} = \frac{\exp(\delta_{jct} / \lambda) \cdot \left(\sum_{k \in G} \exp(\delta_{kct} / \lambda)\right)^{\lambda-1}}{1 + \left(\sum_{k \in G} \exp(\delta_{kct} / \lambda)\right)^\lambda}$$

The share of the outside option is

$$s_{0ct} = \frac{1}{1 + \left(\sum_{k \in G} \exp(\delta_{kct} / \lambda)\right)^\lambda}$$

Taking logs and using the outside option as the base, the nested logit demand equation can be written as:

$$\ln(s_{jct}) - \ln(s_{0ct}) = \beta_j + \alpha p_{jct} + (1-\lambda)\ln(s_{j|G,ct}) + \xi_{jct}.$$

Here, $(1-\lambda)$ is the coefficient on the within-nest share $\ln(s_{j|G,ct})$. When $\lambda = 1$, this term drops out and we get the standard multinomial logit model (no nesting). When $\lambda < 1$, products within the nest are closer substitutes than products outside the nest.

### Question 2

In [9]:
import pandas as pd
import numpy as np
from linearmodels.iv import IV2SLS

data = pd.read_csv('airlines_long_2.csv')

data['share'] = data['passengers'] / data['mkt_size']

# outside good
data['share_outside'] = data.groupby(['route_id', 'quarter'])['share'].transform(lambda x: 1 - x.sum())
# inside nest share
data['share_nest'] = data.groupby(['route_id', 'quarter'])['share'].transform('sum')

# s_{j|G,ct} = s_jct / s_G,ct
data['share_cond'] = data['share'] / data['share_nest']

# inside nest size
data['nest_size'] = data.groupby(['route_id', 'quarter'])['airline_id'].transform('count')

# ln(s_jct) - ln(s_0ct)
data['y'] = np.log(data['share'].clip(1e-10)) - np.log(data['share_outside'].clip(1e-10))

# ln(s_{j|G,ct})
data['log_share_cond'] = np.log(data['share_cond'].clip(1e-10))

data = data.dropna(subset=['y', 'price', 'avg_hub', 'airline_id', 'log_share_cond', 'nest_size']).copy().reset_index(drop=True)

# airline dummies for β_j
airline_dummies = pd.get_dummies(data['airline_id'], prefix='airline', drop_first=True)
# add constant!!!
exog = airline_dummies.copy()
exog['const'] = 1.0

# endogenous regressors: price and log_share_cond (both depend on unobserved ξ)
# instruments used: avg_hub and nest_size
y = data['y']
endog = data[['price', 'log_share_cond']]
instruments = data[['avg_hub', 'nest_size']]

# 2SLS: y = β_j + α*price + ξ - λ*log_share_cond
model = IV2SLS(y, exog, endog, instruments).fit()
print(model.summary)

print(f"\n\n\nα (price coefficient): {model.params['price']:.6f}")
print(f"λ (nest correlation parameter): {1 - model.params['log_share_cond']:.6f}")

                          IV-2SLS Estimation Summary                          
Dep. Variable:                      y   R-squared:                      0.1177
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1172
No. Observations:               15858   F-statistic:                    2421.1
Date:                Mon, Mar 02 2026   P-value (F-stat)                0.0000
Time:                        12:20:47   Distribution:                  chi2(9)
Cov. Estimator:                robust                                         
                                                                              
                               Parameter Estimates                                
                Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------------------------------------------------
airline_2         -0.6081     0.0819    -7.4268     0.0000     -0.7686     -0.4476
airline_3          0.7722     0.0926

### Question 3

$\ln(s_{j|G,ct})$ is

$$\hat\gamma = 0.5808,$$

and in our specification

$$\ln(s_{jct}) - \ln(s_{0ct}) = \beta_j + \alpha p_{jct} + (1-\lambda)\ln(s_{j|G,ct}) + \xi_{jct},$$

so

$$\hat\lambda = 1 - \hat\gamma \approx 1 - 0.5808 = 0.42.$$

A value of $\lambda = 0.42$ lies in $(0,1)$, so the nested logit is valid and there is non-trivial correlation in tastes within the airline nest. Intuitively, inside options (other airlines) are closer substitutes to each other than to the outside option (not flying): when one airline raises its price, consumers are more likely to switch to another carrier than to drop out of the market. Because $\lambda$ is well below 1, the nesting structure matters—airlines are a distinct group of options, but they are not perfect substitutes (there is still differentiation in brand, network, service, etc.).

### Question 4

The profit function for airline $j$ in route–quarter $(c,t)$ is

$$\pi_{jct} = (p_{jct} - mc_{jct}) \cdot s_{jct} \cdot M_{ct}.$$

Maximize:

$$0 = s_{jct} + (p_{jct} - mc_{jct}) \cdot \frac{\partial s_{jct}}{\partial p_{jct}}.$$

Or

$$p_{jct} - mc_{jct} = - \frac{s_{jct}}{\frac{\partial s_{jct}}{\partial p_{jct}}}.$$

For the nested logit with a single inside nest $G$, the own-price derivative of market share is

$$\frac{\partial s_{jct}}{\partial p_{jct}} = \left[ \frac{\alpha}{\lambda} \, s_{jct} \, \Big( 1 - (1-\lambda) s_{j|G,ct} - \lambda s_{jct} \Big) \right].$$

Plugging this into the FOC:

$$mc_{jct} = p_{jct} + s_{jct} \cdot \left[ \frac{\alpha}{\lambda} \, s_{jct} \, \Big( 1 - (1-\lambda)s_{j|G,ct} - \lambda s_{jct} \Big) \right]^{-1}.$$

In [19]:
import numpy as np

# Parameters from Question 2
alpha = float(model.params["price"])               # price coefficient α
coef_logshare = float(model.params["log_share_cond"])  # this is (1 - λ)
lambda_hat = 1.0 - coef_logshare                   # recover λ

# Own-price derivative under nested logit:
# ds_j/dp_j = (α/λ) * s_j * [1 - (1-λ) * s_{j|G} - λ * s_j]
data["ds_dp_own"] = (
    (alpha / lambda_hat)
    * data["share"]
    * (1.0 - (1.0 - lambda_hat) * data["share_cond"] - lambda_hat * data["share"])
)

# Implied marginal cost from FOC: mc_j = p_j + s_j / (ds_j/dp_j)
data["mc"] = data["price"] + data["share"] / data["ds_dp_own"]

# Markups
data["markup"] = data["price"] - data["mc"]

print("Summary of implied marginal costs (all airlines):")
print(data["mc"].describe().round(3))

print("\nSummary of implied markups:")
print(data["markup"].describe().round(3))

print("\nImplied MC summary by airline:")
print(
    data.groupby("airline_id").agg(
        mc_mean=("mc", "mean"),
        mc_median=("mc", "median"),
        mc_std=("mc", "std"),
        price_mean=("price", "mean"),
        price_std=("price", "std"),
        n_obs=("mc", "count"),
    ).round(2)
)

Summary of implied marginal costs (all airlines):
count    15858.000
mean       131.869
std        133.078
min        -49.857
25%         41.491
50%        104.081
75%        193.530
max       2318.080
Name: mc, dtype: float64

Summary of implied markups:
count    15858.000
mean        31.273
std         11.148
min         19.854
25%         21.056
50%         26.355
75%         46.144
max         72.684
Name: markup, dtype: float64

Implied MC summary by airline:
            mc_mean  mc_median  mc_std  price_mean  price_std  n_obs
airline_id                                                          
1            141.22     113.62  117.00      172.12     117.31   1805
2            132.62     101.92  118.31      166.04     117.12   1198
3            186.45     161.90  148.40      219.24     148.79   1602
4            170.06     150.59  123.17      206.47     121.65   1398
5            129.18      88.14  129.53      155.21     128.23   4165
6             51.89      22.00   90.34       88.

### Question 5

Suppose the merged AA–DL firm owns a collection of flights $J_M$ in a given market (route–quarter). Let total market size be $M_{ct}$. Its profit in that market is

$$
\Pi_{ct} = \sum_{j \in J_M} (p_{jct} - mc_{jct}) \, s_{jct} \, M_{ct}.
$$

$$
0 = \frac{\partial \Pi_{ct}}{\partial p_{ict}}
  = s_{ict} + \sum_{j \in J_M} (p_{jct} - mc_{jct}) \frac{\partial s_{jct}}{\partial p_{ict}}.
$$

$$
 s_{ict} + (p_{ict} - mc_{ict})\frac{\partial s_{ict}}{\partial p_{ict}}
   + \sum_{k \in J_M, k \neq i} (p_{kct} - mc_{kct}) \frac{\partial s_{kct}}{\partial p_{ict}} = 0.
$$

$$
(p_{ict} - mc_{ict}) \frac{\partial s_{ict}}{\partial p_{ict}}
  = - s_{ict} - \sum_{k \in J_M, k \neq i} (p_{kct} - mc_{kct}) \frac{\partial s_{kct}}{\partial p_{ict}}.
$$

$$
 p_{ict}
  = mc_{ict}
    - \frac{s_{ict} + \sum_{k \in J_M, k \neq i} (p_{kct} - mc_{kct}) \frac{\partial s_{kct}}{\partial p_{ict}}}
           {\frac{\partial s_{ict}}{\partial p_{ict}}}.
$$

In the special case where the merged firm owns exactly two flights on the route, one AA and one DL:

- For AA:

$$
 p_{AA} = mc_{AA} - \frac{s_{AA} + (p_{DL} - mc_{DL}) \frac{\partial s_{DL}}{\partial p_{AA}}}
                            {\frac{\partial s_{AA}}{\partial p_{AA}}},
$$

- For DL:

$$
 p_{DL} = mc_{DL} - \frac{s_{DL} + (p_{AA} - mc_{AA}) \frac{\partial s_{AA}}{\partial p_{DL}}}
                            {\frac{\partial s_{DL}}{\partial p_{DL}}}.
$$

In [26]:
# AA–DL merger under nested logit

import numpy as np
import pandas as pd

sim = data.copy()
sim["price_pre"] = sim["price"]

alpha_hat = float(model.params["price"])
coef_logshare = float(model.params["log_share_cond"])  # (1 - λ)
lambda_hat = 1.0 - coef_logshare

# Recover δ̂_j net of price from the estimated demand equation
# y = log(s_j) - log(s_0) = δ_j + (1-λ) log(s_{j|G})  ⇒  δ_j = y - (1-λ) log(s_{j|G})
sim["delta_pre"] = sim["y"] - coef_logshare * sim["log_share_cond"]

AA_ID, DL_ID = 1, 3
sim["is_merged"] = sim["airline_id"].isin([AA_ID, DL_ID]).astype(int)

# Drop markets with zero prices or negative costs
bad = (sim["price_pre"] <= 0) | (sim["mc"] < 0)
bad_mkts = sim.loc[bad, ["route_id", "quarter"]].drop_duplicates()
if not bad_mkts.empty:
    sim = sim.merge(bad_mkts.assign(_drop=1), on=["route_id", "quarter"], how="left")
    sim = sim[sim["_drop"].isna()].drop(columns=["_drop"]).copy()

def recompute_shares(df: pd.DataFrame, alpha: float, lam: float) -> pd.DataFrame:
    # Update δ using only the price part
    df["delta"] = df["delta_pre"] + alpha * (df["price"] - df["price_pre"])
    df["delta"] = np.clip(df["delta"], a_min=None, a_max=700)

    df["exp_delta_lam"] = np.exp(df["delta"] / lam)
    df["D_G"] = df.groupby(["quarter", "route_id"])['exp_delta_lam'].transform('sum')

    df["s_j_given_G"] = df["exp_delta_lam"] / df["D_G"]
    df["s_G"] = (df["D_G"] ** lam) / (1.0 + df["D_G"] ** lam)
    df["s_j"] = df["s_j_given_G"] * df["s_G"]
    return df

def best_response(df: pd.DataFrame, alpha: float, lam: float) -> pd.DataFrame:
    a = alpha
    lam_ = lam

    df["ds_dp_ii"] = (
        (a / lam_)
        * df["s_j"]
        * (1.0 - (1.0 - lam_) * df["s_j_given_G"] - lam_ * df["s_j"])
    )

    df["margin"] = df["price"] - df["mc"]
    df["diversion_weight"] = (1.0 - lam_) * df["s_j_given_G"] + lam_ * df["s_j"]
    df["mw_diversion"] = df["margin"] * df["diversion_weight"] * df["is_merged"]

    df["sum_mw"] = df.groupby(["quarter", "route_id"])['mw_diversion'].transform('sum')
    df["partner_term"] = df["sum_mw"] - df["mw_diversion"]

    df["cross_effect"] = -(a / lam_) * df["s_j"] * df["partner_term"]

    # p = mc − s / ds_dp
    p_single = df["mc"] - df["s_j"] / df["ds_dp_ii"]

    # p = mc − (s + cross_effect) / ds_dp
    p_merged = df["mc"] - (df["s_j"] + df["cross_effect"]) / df["ds_dp_ii"]

    df["price_br"] = np.where(df["is_merged"] == 1, p_merged, p_single)
    return df

# Iterate best responses to a fixed point
sim = recompute_shares(sim, alpha_hat, lambda_hat)

max_iter = 3000
step = 0.05
tol = 1e-6
it = 0
max_diff = np.inf

while it < max_iter and max_diff > tol:
    sim = recompute_shares(sim, alpha_hat, lambda_hat)
    sim = best_response(sim, alpha_hat, lambda_hat)

    max_diff = np.max(np.abs(sim["price_br"] - sim["price"]))
    sim["price"] = step * sim["price_br"] + (1.0 - step) * sim["price"]
    it += 1

print(f"Converged in {it} iterations (max |Δp| = {max_diff:.3g}).")

# Final shares and % price changes
sim = recompute_shares(sim, alpha_hat, lambda_hat)
sim["pct_change"] = 100.0 * (sim["price"] - sim["price_pre"]) / sim["price_pre"]

# Average change for merged airlines across all markets
mask_merged_airlines = sim["airline_id"].isin([AA_ID, DL_ID])
avg_change_all = (
    sim[mask_merged_airlines]
    .groupby("airline_id", as_index=False)["pct_change"]
    .mean()
    .rename(columns={"pct_change": "avg_pct_change"})
    .sort_values("airline_id")
)
avg_change_all["avg_pct_change"] = avg_change_all["avg_pct_change"].round(3)

print("\nAverage % price change for merged airlines (all):")
print(avg_change_all)

# Average change for overlapping AA–DL markets only
overlap = (
    sim.groupby(["quarter", "route_id"])['airline_id']
    .apply(lambda ids: ids.isin([AA_ID, DL_ID]).sum() == 2)
    .reset_index(name="both_present")
)

sim_ov = sim.merge(overlap[overlap["both_present"]], on=["quarter", "route_id"], how="inner")

avg_change_ov = (
    sim_ov[sim_ov["airline_id"].isin([AA_ID, DL_ID])]
    .groupby("airline_id", as_index=False)["pct_change"]
    .mean()
    .rename(columns={"pct_change": "avg_pct_change"})
    .sort_values("airline_id")
)
avg_change_ov["avg_pct_change"] = avg_change_ov["avg_pct_change"].round(3)

print("\nAverage % price change for merged airlines (overlap):")
print(avg_change_ov)

Converged in 543 iterations (max |Δp| = 9.87e-07).

Average % price change for merged airlines (all):
   airline_id  avg_pct_change
0           1           1.162
1           3           2.509

Average % price change for merged airlines (overlap):
   airline_id  avg_pct_change
0           1           3.686
1           3           7.796


Overall, the nested logit merger simulation predicts smaller average price increases for AA and DL than the simple logit model. Once we account for the nesting parameter ($\alpha/\lambda$), consumers are effectively more price sensitive, and with $\lambda < 0.5$ they substitute strongly to other airlines rather than to the outside option. Because AA and DL are not very close substitutes, a price increase on one mostly shifts demand to rival carriers instead of its merger partner. As a result, the merged firm has weaker incentives to raise prices, so the nested logit model generates more modest merger price effects.

